# Project Lab I: Operations Analyzer, Sales Dashboard, Excel Dashboard

Notebook for **Chapter 16** of the *analyst* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/analyst/ch16.html

Tip: change a number, predict the output, then run the cell again.

## 16.1 How to Run a Project in This Area

## 16.2 Project: Operations Results Analyzer

In [ ]:
# src/make_data.py
import numpy as np, pandas as pd
rng = np.random.default_rng(42)
n = 3000
start = pd.Timestamp("2025-03-01")
ts = start + pd.to_timedelta(np.sort(rng.integers(0, 14*24*60, n)), unit="m")
kind = rng.choice(["standard", "express", "international"], n, p=[0.5, 0.35, 0.15])
med = pd.Series(kind).map({"standard": 40, "express": 30, "international": 90}).to_numpy()
day = (ts - start).days.to_numpy()
mins = np.exp(rng.normal(np.log(med), 0.5, n)) * (1 + 0.01 * day)   # slow upward drift
status = np.where(rng.random(n) < (0.02 + 0.002*day), "error", "ok")
df = pd.DataFrame({"order_id": np.arange(1, n+1), "ts": ts, "order_type": kind,
                   "minutes": mins.round(1), "status": status})
df.to_csv("orders.csv", index=False)
print(df.head(3)); print(df.shape)

In [ ]:
import sqlite3, pandas as pd
df = pd.read_csv("orders.csv", parse_dates=["ts"])
con = sqlite3.connect("ops.db")
df.assign(ts=df.ts.astype(str)).to_sql("orders", con, if_exists="replace", index=False)
print(pd.read_sql("""
SELECT COUNT(*) AS n, ROUND(AVG(minutes),1) AS avg_min,
       ROUND(100.0*SUM(status='error')/COUNT(*),2) AS error_pct
FROM orders""", con))

      n  avg_min  error_pct
0  3000     53.3       3.7


In [ ]:
q_pct = """
WITH ranked AS (
  SELECT minutes, ROW_NUMBER() OVER (ORDER BY minutes) AS rn, COUNT(*) OVER () AS n
  FROM orders WHERE status='ok')
SELECT
 MIN(CASE WHEN rn >= 0.50*n THEN minutes END) AS p50,
 MIN(CASE WHEN rn >= 0.90*n THEN minutes END) AS p90,
 MIN(CASE WHEN rn >= 0.95*n THEN minutes END) AS p95,
 MIN(CASE WHEN rn >= 0.99*n THEN minutes END) AS p99
FROM ranked"""
print(pd.read_sql(q_pct, con))

    p50   p90    p95    p99
0  42.3  99.0  125.1  211.0


In [ ]:
ok = df[df.status=="ok"].minutes
print(ok.quantile([.5,.9,.95,.99]).round(1).to_dict())
print(ok.quantile([.5,.9,.95,.99], interpolation="higher").round(1).to_dict())

{0.5: 42.3, 0.9: 98.9, 0.95: 124.9, 0.99: 210.4}
{0.5: 42.3, 0.9: 99.0, 0.95: 125.1, 0.99: 211.0}


In [ ]:
by = df[df.status == "ok"].groupby("order_type")["minutes"].agg(
    n="size", avg="mean", median="median",
    p90=lambda s: s.quantile(.90), p95=lambda s: s.quantile(.95)).round(1)
by["error_pct"] = df.groupby("order_type")["status"].apply(lambda s: (s=="error").mean()*100).round(2)
print(by)

                  n    avg  median    p90    p95  error_pct
order_type                                                  
express        1003   36.9    31.9   63.7   76.5       3.84
international   428  108.8    95.5  183.6  218.1       3.82
standard       1458   48.0    42.4   80.3   98.7       3.57


In [ ]:
daily = pd.read_sql("""SELECT DATE(ts) AS day, COUNT(*) AS n, ROUND(AVG(minutes),1) AS avg_min,
 ROUND(100.0*SUM(status='error')/COUNT(*),1) AS err_pct FROM orders GROUP BY DATE(ts)""", con)
import numpy as np
x = np.arange(len(daily))
for c in ["avg_min","err_pct"]:
    print(c, "slope per day", round(np.polyfit(x, daily[c], 1)[0], 3))

avg_min slope per day 0.45
err_pct slope per day 0.34


In [ ]:
assert df.order_id.is_unique and df.minutes.gt(0).all()
assert len(df[df.status=="ok"]) + (df.status=="error").sum() == len(df)
print("checks passed")

checks passed


## 16.3 Project: Sales Dashboard Design in Power BI

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(7)
products = pd.DataFrame({
  "ProductKey": range(1, 9),
  "Product": ["Notebook","Pen Pack","Backpack","Desk Lamp","Headphones","Water Bottle","Calculator","Planner"],
  "Category": ["Paper","Paper","Bags","Home","Tech","Home","Tech","Paper"],
  "UnitPrice": [120, 60, 900, 650, 1500, 300, 450, 250],
  "UnitCost":  [70, 30, 520, 380, 1000, 150, 300, 120]})
customers = pd.DataFrame({
  "CustomerKey": range(1, 201),
  "Region": rng.choice(["North","South","East","West"], 200),
  "Segment": rng.choice(["Consumer","Business"], 200, p=[0.7, 0.3])})
dates = pd.DataFrame({"Date": pd.date_range("2024-01-01", "2025-12-31")})
n = 6000
fact = pd.DataFrame({
  "OrderID": np.arange(1, n+1),
  "Date": pd.to_datetime(rng.choice(dates.Date, n)),
  "CustomerKey": rng.integers(1, 201, n),
  "ProductKey": rng.integers(1, 9, n),
  "Qty": rng.integers(1, 6, n)})
fact = fact.sort_values("Date").reset_index(drop=True)
fact["OrderID"] = range(1, n+1)
fact.to_csv("fact_sales.csv", index=False); products.to_csv("dim_product.csv", index=False)
customers.to_csv("dim_customer.csv", index=False); dates.to_csv("dim_date.csv", index=False)
m = fact.merge(products, on="ProductKey")
m["Sales"] = m.Qty*m.UnitPrice; m["Cost"] = m.Qty*m.UnitCost
m["Year"] = m.Date.dt.year
print(len(fact), "rows;", fact.Date.min().date(), "to", fact.Date.max().date())
print(m.groupby("Year")[["Sales","Cost"]].sum())
t = m.groupby("Year")["Sales"].sum()
print("YoY %", round((t[2025]/t[2024]-1)*100, 2))
print("Margin %", round((1-m.Cost.sum()/m.Sales.sum())*100, 2))
print(m.groupby("Category").Sales.sum().sort_values(ascending=False))

## 16.4 Project: Excel Dashboard with Power Query and PivotTables

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(13)
tot = []
for m in ["2025-01","2025-02","2025-03"]:
    n = 120
    df = pd.DataFrame({
      "Date": rng.choice(pd.date_range(m+"-01", periods=28), n),
      "Store": rng.choice(["Pune","Chennai","Jaipur"], n),
      "Item": rng.choice(["Rice","Oil","Tea","Soap"], n),
      "Units": rng.integers(1, 20, n),
      "Price": rng.choice([60, 140, 250, 35], n)})
    df["Amount"] = df.Units*df.Price
    df.to_csv(f"raw/sales_{m}.csv", index=False)
    tot.append(df)
a = pd.concat(tot)
print(len(a), a.Amount.sum())
a["Month"] = a.Date.dt.month
print(a.pivot_table(index="Store", columns="Month", values="Amount", aggfunc="sum", margins=True))

360 427270
Month         1       2       3     All
Store                                  
Chennai   44875   43870   57525  146270
Jaipur    44870   38270   46670  129810
Pune      47015   50305   53870  151190
All      136760  132445  158065  427270


## Chapter recap

## Practice set

**1. (Easy)** Twenty order times in minutes, sorted: 5, 7, 8, 9, 10, 11, 12, 12, 13, 14, 15, 16, 18, 20, 22, 25, 30, 41, 60, 95. Find the mean, the median, and the nearest-rank 90th percentile.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/analyst/ch16.html